In [3]:
import nanopub
print(nanopub.__version__)

2.1.0


In [1]:
import pandas as pd
flora = pd.read_csv("https://raw.githubusercontent.com/forrtproject/fred-data/refs/heads/main/output/flora.csv")


# FLoRA filtering

In [ ]:
flora_filtered = flora[flora["outcome"].isin(["successful", "failed", "mixed"])]




outcome
successful    1019
failed         989
mixed          625
Name: count, dtype: int64

In [64]:
# multi-study papers: several flora rows share one paper identifier but each has its own url_r.
# For these, work = url_r and the paper (doi_r or alt_identifier_r) goes into `collection`.

def shared_id_unique_url(df, key):
    """Rows whose `key` is shared across several rows, where every row in that
    group has a url_r and no two rows in the group have the same url_r."""
    d = df[df[key].notna()]
    g = d.groupby(key)["url_r"]
    qualifies = (
        (g.transform("size") > 1)                          # key shared across rows
        & (g.transform(lambda s: s.notna().all()))         # every row has a url_r
        & (g.transform("nunique") == g.transform("size"))  # url_r all different
    )
    return d[qualifies]

# A: doi_r shared across rows, each with a unique url_r
group_a = shared_id_unique_url(flora, "doi_r")
# B: no doi_r, alt_identifier_r shared across rows, each with a unique url_r
group_b = shared_id_unique_url(flora[flora["doi_r"].isna()], "alt_identifier_r")

multi_studies = pd.concat([group_a.assign(match="doi_r"),
                           group_b.assign(match="alt_identifier_r")])
multi_study_ids = set(multi_studies["id"])

print(len(group_a), "rows via doi_r,", len(group_b), "rows via alt_identifier_r")
multi_studies.sort_values(["doi_r", "alt_identifier_r"])[["id", "match", "doi_r", "alt_identifier_r", "url_r"]]
multi_studies[multi_studies["url_r"] == "https://osf.io/bfs6p"][["id", "match", "doi_r", "alt_identifier_r", "url_r"]   ]

61 rows via doi_r, 179 rows via alt_identifier_r


,id,match,doi_r,alt_identifier_r,url_r
2337,FLORA-002635,alt_identifier_r,NaN,10.1038/s41586-025-10078-y,https://osf.io/bfs6p


# using template to publish nanopub

In [62]:
test = flora_filtered[
    (flora_filtered["doi_r"] == "10.1080/713755986")
    | (flora_filtered["url_r"] == "https://osf.io/bfs6p")
]
test[["id", "doi_r", "alt_identifier_r", "url_r", "outcome"]]
test.head()

,id,id_md5,doi_o,alt_identifier_o,doi_o_hash,title_o,author_o,journal_o,year_o,volume_o,...,abstract_r,outcome_computation,outcome_computational_quote,out_quote_computational_source,outcome_robustness,outcome_robustness_quote,out_quote_robust_source,flora_id,source_display_id,source_record_id
1001,FLORA-001011,a4b2fc10fec1dcf3be3ce75c3a4da4d1,10.1006/jesp.1996.0005,NaN,9d6,The Automatic Evaluation Effect: Unconditional...,"[{""given"":""John A."",""family"":""Bargh"",""sequence...",Journal of Experimental Social Psychology,1996,32,...,Recent work has found an affective priming eff...,NaN,NaN,NaN,NaN,NaN,NaN,REPL-001156,REPL-001156,3e3d4f2b-beb1-4437-87cb-437270880ce1
2337,FLORA-002635,029e1279d9cd5083d4a953d2efc706d3,10.1037/pspp0000113,NaN,eb0,Narcissism and romantic relationships: The dif...,"[{""given"":""Stefanie N."",""family"":""Wurst"",""sequ...",Journal of Personality and Social Psychology,2017,112,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REPL-000420,REPL-000420,95786a9a-8112-4f8b-b6ee-ee090ae096da


# required fields

In [6]:
import requests
import pandas as pd
from rdflib import Dataset, Namespace, RDF, RDFS

TEMPLATES = [
    "https://w3id.org/np/RAUO3ykIwaMXNs8Do-pGujh1ELhmux_ajG6sAO_0Jk4EA",
    "https://w3id.org/np/RAZR5s1rLpW8GTHXorx6zZL1cIUPxSPjzD32lL4aDUh2k",
    "https://w3id.org/np/RAINLsq3Dj9_JhPfJqtmozdSCWufYB0NcOXOHd2BMhKnk"
]

NT = Namespace("https://w3id.org/np/o/ntemplate/")

def template_fields(template_uri):
    """Return a table of the fields (placeholders) in a nanopub template."""
    r = requests.get(template_uri, headers={"Accept": "application/trig"})
    r.raise_for_status()
    ds = Dataset()
    ds.parse(data=r.text, format="trig")
    g = next(c for c in ds.graphs() if str(c.identifier).endswith("/assertion"))
    tmpl = next(g.subjects(RDF.type, NT.AssertionTemplate))

    def walk(st, optional=False, repeatable=False):
        types = set(g.objects(st, RDF.type))
        optional |= NT.OptionalStatement in types
        repeatable |= NT.RepeatableStatement in types
        subs = list(g.objects(st, NT.hasStatement))
        if subs:  # grouped statement
            for s in subs:
                yield from walk(s, optional, repeatable)
        else:
            yield st, optional, repeatable

    fields = {}
    for st in g.objects(tmpl, NT.hasStatement):
        for leaf, opt, rep in walk(st):
            for pos in (RDF.subject, RDF.predicate, RDF.object):
                node = g.value(leaf, pos)
                ptypes = [t for t in g.objects(node, RDF.type)
                          if str(t).startswith(str(NT)) and "Placeholder" in str(t)]
                if not ptypes:
                    continue
                f = fields.setdefault(node, {
                    "field": str(node).rsplit("/", 1)[-1],
                    "label": str(g.value(node, RDFS.label) or ""),
                    "type": str(ptypes[0]).rsplit("/", 1)[-1],
                    "required": False,
                    "repeatable": False,
                })
                f["required"] |= not opt
                f["repeatable"] |= rep
    return pd.DataFrame(fields.values())

template_fields(TEMPLATES[2])



,field,label,type,required,repeatable
0,work,DOI (https://doi.org/10...) or other URL of th...,ExternalUriPlaceholder,True,True
1,cites,select the citation type,RestrictedChoicePlaceholder,True,True
2,cited,DOI (https://doi.org/10...) or other URL of th...,ExternalUriPlaceholder,True,True
3,outcome,Quote the replication/reproduction outcome,LongLiteralPlaceholder,True,False
4,collection,DOI (https://doi.org/10...) or other URL of th...,ExternalUriPlaceholder,False,False


# Nanopubs

In [ ]:
from rdflib import Graph, Literal, Namespace, URIRef, RDF, RDFS
from nanopub import Nanopub, NanopubConf, load_profile

TEMPLATE = URIRef("https://w3id.org/np/RAINLsq3Dj9_JhPfJqtmozdSCWufYB0NcOXOHd2BMhKnk")
CITO  = Namespace("http://purl.org/spar/cito/")
FORRT = Namespace("https://w3id.org/sciencelive/o/terms/")
NT    = Namespace("https://w3id.org/np/o/ntemplate/")
DCT   = Namespace("http://purl.org/dc/terms/")

OUTCOME_TO_CITO = {
    "successful": CITO.confirms,
    "failed":     CITO.refutes,
    "mixed":      CITO.discusses,
}

def doi_uri(doi):
    """Turn flora identifiers into URIs: bare DOIs -> https://doi.org/..., other bare links -> https://..."""
    doi = str(doi).strip()
    if doi.startswith("http"):
        return URIRef(doi)
    if doi.startswith("10."):
        return URIRef("https://doi.org/" + doi)
    return URIRef("https://" + doi)

def require(row, *cols):
    for col in cols:
        if pd.isna(row[col]):
            raise ValueError(f"Required column {col!r} is empty")

def build_nanopub(row, conf, multi_ids=frozenset()):
    outcome = str(row["outcome"]).strip().lower()
    if outcome not in OUTCOME_TO_CITO:
        raise ValueError(f"Can't map outcome {row['outcome']!r} to a CiTO relation")
    require(row, "doi_o", "outcome_quote")

    if row["id"] in multi_ids:
        # multi-study paper: the individual study is the work, the paper is the collection
        require(row, "url_r")
        work = doi_uri(row["url_r"])
        paper = row["doi_r"] if pd.notna(row["doi_r"]) else row["alt_identifier_r"]
        collection = doi_uri(paper)
    else:
        require(row, "doi_r")
        work = doi_uri(row["doi_r"])
        collection = None
    cited = doi_uri(row["doi_o"])

    assertion = Graph()
    assertion.bind("cito", CITO)
    assertion.bind("forrt", FORRT)
    assertion.bind("dct", DCT)
    assertion.add((work, RDF.type, FORRT["FORRT-Replication-Study"]))               # st01
    assertion.add((work, OUTCOME_TO_CITO[outcome], cited))                          # st02
    assertion.add((work, FORRT["FORRT-Replication-Outcome"],
                   Literal(str(row["outcome_quote"]).strip())))                     # st03
    if collection is not None:                                                      # st04 (optional)
        assertion.add((work, DCT.isPartOf, collection))

    np = Nanopub(conf=conf, assertion=assertion)
    this = np.metadata.namespace[""]
    np.pubinfo.bind("nt", NT)
    np.pubinfo.add((this, NT.wasCreatedFromTemplate, TEMPLATE))
    np.pubinfo.add((this, RDFS.label, Literal(f"Replication outcome of: {work}")))
    return np

np_conf = NanopubConf(
    profile=load_profile(),
    use_test_server=True,
    add_prov_generated_time=True,
    attribute_assertion_to_profile=True,
    attribute_publication_to_profile=True,
)


@prefix cito: <http://purl.org/spar/cito/> .
@prefix forrt: <https://w3id.org/sciencelive/o/terms/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix orcid: <https://orcid.org/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://purl.org/nanopub/temp/np/Head> {
    <http://purl.org/nanopub/temp/np/> a np:Nanopublication ;
        np:hasAssertion <http://purl.org/nanopub/temp/np/assertion> ;
        np:hasProvenance <http://purl.org/nanopub/temp/np/provenance> ;
        np:hasPublicationInfo <http://purl.org/nanopub/temp/np/pubinfo> .
}

<http://purl.org/nanopub/temp/np/pubinfo> {
    <http://purl.org/nanopub/temp/np/> rdfs:label "Replication outcome of: https://doi.org/10.1097/mlr.0000000000002256" ;
        prov:generatedAtTime "2026-11-12T15:53:03.613604+01:00"^^xsd:dateTime ;
        prov:wasAttributedTo orcid:0000-0

# build nanopubs for many flora entries
- start with a small batch on the test server
- rows with an unmappable outcome or missing required fields are skipped and reported
- duplicate rows (same work, original and outcome) are dropped so the same claim isn't published twice
- rows in `multi_study_ids` get work = `url_r` and collection = `doi_r` / `alt_identifier_r`

In [66]:
batch = test   

# drop duplicate claims (same replication, original and outcome).
# url_r and alt_identifier_r are included so multi-study rows sharing a doi_r aren't dropped.
dup_cols = ["doi_r", "alt_identifier_r", "url_r", "doi_o", "outcome"]
n_dups = batch.duplicated(dup_cols).sum()
batch = batch.drop_duplicates(dup_cols)
print(f"Dropped {n_dups} duplicate rows")

nanopubs_built = []   # (flora id, Nanopub) pairs
skipped = []          # (flora id, reason) pairs

for _, row in batch.iterrows():
    try:
        nanopubs_built.append((row["id"], build_nanopub(row, np_conf, multi_study_ids)))
    except ValueError as e:
        skipped.append((row["id"], str(e)))

n_multi = sum(i in multi_study_ids for i, _ in nanopubs_built)
print(f"Built {len(nanopubs_built)} nanopubs ({n_multi} multi-study), skipped {len(skipped)} rows")
skipped_df = pd.DataFrame(skipped, columns=["id", "reason"])
skipped_df.reason.value_counts()

Dropped 0 duplicate rows
Built 2 nanopubs (1 multi-study), skipped 0 rows


Series([], Name: count, dtype: int64)

In [45]:
# inspect a few before publishing
for flora_id, np_ in nanopubs_built:
    print(flora_id)
    print(np_.assertion.serialize(format="turtle"))

FLORA-000002
@prefix cito: <http://purl.org/spar/cito/> .
@prefix forrt: <https://w3id.org/sciencelive/o/terms/> .

<https://doi.org/10.3758/s13423-014-0582-9> a forrt:FORRT-Replication-Study ;
    cito:confirms <https://doi.org/10.1037/h0080111> ;
    forrt:FORRT-Replication-Outcome "“Thus, the present experiment replicates the earlier data reported by Rayner et al. (1986), indicating evidence for an orthographic preview benefit but a lack of semantic preview benefit in reading English.”" .


FLORA-000003
@prefix cito: <http://purl.org/spar/cito/> .
@prefix forrt: <https://w3id.org/sciencelive/o/terms/> .

<https://doi.org/10.1136/jmg.2009.075911> a forrt:FORRT-Replication-Study ;
    cito:confirms <https://doi.org/10.1093/hmg/ddp542> ;
    forrt:FORRT-Replication-Outcome "We here replicate the association with multiple sclerosis in a Belgian study population of 791 patients and 1098 controls." .




# publish (not run yet)
- every published nanopub is logged to a CSV (separate files for test and real server)
- on a rerun, flora ids already in the log are skipped, so nothing is published twice
- short pause between publishes so the server isn't flooded

In [68]:
import csv, os, time
from datetime import datetime, timezone
from nanopub.definitions import TEST_NANOPUB_REGISTRY_URL

# the library swaps in the test server internally, so work out the real target here
SERVER = TEST_NANOPUB_REGISTRY_URL if np_conf.use_test_server else np_conf.use_server
LOG_FILE = "published_test.csv" if np_conf.use_test_server else "published.csv"
LOG_COLS = ["id", "nanopub", "server", "published_at"]

def already_published():
    if not os.path.exists(LOG_FILE):
        return set()
    return set(pd.read_csv(LOG_FILE)["id"])

def publish_batch(built, pause=0.5):
    done = already_published()
    todo = [(i, n) for i, n in built if i not in done]
    print(f"{len(built) - len(todo)} already published, publishing {len(todo)} to {SERVER}")

    new_file = not os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_COLS)
        if new_file:
            writer.writeheader()
        for flora_id, np_ in todo:
            try:
                np_.publish()
            except Exception as e:
                print(f"{flora_id}: failed - {e}")
                continue
            writer.writerow({
                "id": flora_id,
                "nanopub": np_.source_uri,
                "server": SERVER,
                "published_at": datetime.now(timezone.utc).isoformat(),
            })
            f.flush()   # write each row immediately, so a crash doesn't lose the log
            print(f"{flora_id}: {np_.source_uri}")
            time.sleep(pause)

# uncomment when ready:
publish_batch(nanopubs_built)

0 already published, publishing 2 to https://test.registry.knowledgepixels.com/np/
FLORA-001011: https://w3id.org/np/RAL25a__7YfOp7IQAfllgadFCHNoyKl7qtuuCNM_Zy7Ys
FLORA-002635: https://w3id.org/np/RA_goEffI8la9pnGQynQLQPzRwHgHvLtYwfyMnwLxubKA
